In [1]:
import depth_pro
import torch


model, transform = depth_pro.create_model_and_transforms(device=torch.device("cuda"),precision=torch.float16)
model.eval()

c:\Users\vidha\miniconda3\envs\DEPTH-PRO\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\vidha\Documents\ml-depth-pro\src\depth_pro\depth_pro.py:135: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. W

DepthPro(
  (encoder): DepthProEncoder(
    (patch_encoder): VisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
        (norm): Identity()
      )
      (pos_drop): Dropout(p=0.0, inplace=False)
      (patch_drop): Identity()
      (norm_pre): Identity()
      (blocks): Sequential(
        (0): Block(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (attn): Attention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (q_norm): Identity()
            (k_norm): Identity()
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): LayerScale()
          (drop_path1): Identity()
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linea

In [2]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.backends.cudnn.version())
print(torch.cuda.is_available())



2.5.1+cu121
12.1
90100
True


In [3]:
path = "./data/test1.jpeg"

In [4]:
import torch
print(torch.cuda.is_available())       # True if CUDA is available
print(torch.cuda.current_device())     # Device index
print(torch.cuda.device_count())       # Number of GPUs


True
0
1


In [5]:
image, _, f_px = depth_pro.load_rgb(path)
image = transform(image)
image = image.to("cuda")
prediction = model.infer(image, f_px=f_px)
depth = prediction["depth"]  # Depth in [m]

print(depth)

tensor([[0.8322, 0.8307, 0.8226,  ..., 2.8734, 2.7341, 2.7094],
        [0.8267, 0.8252, 0.8174,  ..., 2.9349, 2.8393, 2.8221],
        [0.8163, 0.8149, 0.8074,  ..., 3.0621, 3.0683, 3.0695],
        ...,
        [0.7563, 0.7559, 0.7539,  ..., 2.4471, 2.4375, 2.4358],
        [0.7582, 0.7581, 0.7580,  ..., 2.3290, 2.2235, 2.2048],
        [0.7591, 0.7593, 0.7601,  ..., 2.2724, 2.1272, 2.1020]],
       device='cuda:0')


In [21]:
from ultralytics import YOLO
import cv2

obj_model = YOLO("./runs/detect/train10/weights/best.pt")
results = obj_model(path, show=True, save=True)


image 1/1 c:\Users\vidha\Documents\VisAssistGlasses\Depth_Model\data\test1.jpeg: 480x640 1 chair, 2 tables, 10.8ms
Speed: 2.4ms preprocess, 10.8ms inference, 2.3ms postprocess per image at shape (1, 3, 480, 640)
Results saved to runs\detect\predict8


In [20]:
from PIL import Image

img = Image.fromarray(results[0].plot())
img_resized = img.resize((img.width // 2, img.height // 2))  # half size
img_resized.show()


In [ ]:


boxes = results[0].boxes  # boxes object
classes = results[0].boxes.cls.cpu().numpy()  # class IDs as numpy array


In [17]:
# COCO class names for YOLOv5/YOLOv8
class_names = ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 
               'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 
               'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 
               'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 
               'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 
               'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 
               'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 
               'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']

for box, cls in zip(boxes, classes):
    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
    cx = int((x1 + x2) / 2)
    cy = int((y1 + y2) / 2)
    
    class_name = class_names[int(cls)]  # map class ID to name
    
    print(f"Detected object: {class_name}")
    print(f"Center of detected object: ({cx}, {cy})")
    print(f"Depth at center: {depth[cy, cx].item()} m")

# Code for using minimum in bounding box (doesn't work in cases where box is much larger than object)
# for box, cls in zip(boxes, classes):
#     # Extract bounding box coordinates
#     x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
    
#     # Convert to integer pixel indices, clamp to depth map size
#     x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])
#     x1 = max(x1, 0)
#     y1 = max(y1, 0)
#     x2 = min(x2, depth.shape[1] - 1)  # width
#     y2 = min(y2, depth.shape[0] - 1)  # height

#     # Slice the depth map for the region inside the bounding box
#     box_depth_region = depth[y1:y2+1, x1:x2+1]  # rows, cols

#     # Get the minimum depth in the box
#     min_depth = box_depth_region.min().item()

#     # Map class ID to name
#     class_name = class_names[int(cls)]

#     print(f"Detected object: {class_name}")
#     print(f"Bounding box: ({x1}, {y1}) to ({x2}, {y2})")
#     print(f"Minimum depth in bounding box: {min_depth:.3f} m")


Detected object: chair
Center of detected object: (1662, 1856)
Depth at center: 3.4772491455078125 m
Detected object: chair
Center of detected object: (1029, 1404)
Depth at center: 6.4222307205200195 m
Detected object: chair
Center of detected object: (2549, 1720)
Depth at center: 4.227035999298096 m
Detected object: dining table
Center of detected object: (2048, 1752)
Depth at center: 5.551126003265381 m
Detected object: dining table
Center of detected object: (1511, 1484)
Depth at center: 5.644118309020996 m
Detected object: dining table
Center of detected object: (1481, 1376)
Depth at center: 4.9949631690979 m
Detected object: bench
Center of detected object: (3175, 2457)
Depth at center: 1.8599687814712524 m
Detected object: chair
Center of detected object: (861, 1314)
Depth at center: 8.317715644836426 m
Detected object: dining table
Center of detected object: (1484, 1299)
Depth at center: 5.464113235473633 m
Detected object: dining table
Center of detected object: (2049, 1493)
De

In [18]:
def generate_llm_prompt(boxes, classes, depth, class_names):
    detected_objects_info = []

    for box, cls in zip(boxes, classes):
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        cx = int((x1 + x2) / 2)
        cy = int((y1 + y2) / 2)
        
        class_name = class_names[int(cls)]  # map class ID to name
        
        # Ensure center coordinates are within depth map bounds
        cx = max(0, min(cx, depth.shape[1] - 1))
        cy = max(0, min(cy, depth.shape[0] - 1))
        
        depth_at_center = depth[cy, cx].item()

        detected_objects_info.append(f"- A {class_name} located at approximate coordinates ({cx}, {cy}) with an estimated depth of {depth_at_center:.2f} meters.")

    if detected_objects_info:
        prompt = "Based on the visual analysis of the environment, the following objects have been detected:\n"
        prompt += "\n".join(detected_objects_info)
        prompt += "\nThis information provides a spatial understanding of the scene. The image shows the user facing directly ahead. Without ever referencing exact coordinates (depth in meters is fine) you will now utilize this information to help guide a blind user who cannot see. \n Talk naturally, like a real-time human assistant.\n Respond with 1-2 sentences. \n If the user asks questions about whether its safe to walk a certain direction, utilize your understanding of how far each object is from them to inform them roughly how many steps they can take."
    else:
        prompt = "No distinct objects were detected in the environment, indicating a potentially clear or unpopulated scene."
    
    return prompt

# Example usage:
llm_prompt = generate_llm_prompt(boxes, classes, depth, class_names)
print(llm_prompt)

Based on the visual analysis of the environment, the following objects have been detected:
- A chair located at approximate coordinates (1662, 1856) with an estimated depth of 3.48 meters.
- A chair located at approximate coordinates (1029, 1404) with an estimated depth of 6.42 meters.
- A chair located at approximate coordinates (2549, 1720) with an estimated depth of 4.23 meters.
- A dining table located at approximate coordinates (2048, 1752) with an estimated depth of 5.55 meters.
- A dining table located at approximate coordinates (1511, 1484) with an estimated depth of 5.64 meters.
- A dining table located at approximate coordinates (1481, 1376) with an estimated depth of 4.99 meters.
- A bench located at approximate coordinates (3175, 2457) with an estimated depth of 1.86 meters.
- A chair located at approximate coordinates (861, 1314) with an estimated depth of 8.32 meters.
- A dining table located at approximate coordinates (1484, 1299) with an estimated depth of 5.46 meters.


In [20]:
from groq import Groq

client = Groq(api_key="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")
completion = client.chat.completions.create(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    messages=[
      {
        "role": "user",
        "content": llm_prompt + "\n Question: " + input("Ask a question: ")
      }
    ],
    temperature=1,
    max_completion_tokens=256,
    top_p=1,
    stream=True,
    stop=None,
)

for chunk in completion:
    print(chunk.choices[0].delta.content or "", end="")

The closest chair to you appears to be about 3.48 meters away. You can probably take around 5-6 steps forward before reaching it.

In [ ]:
# import cv2
# from ultralytics import YOLO
# import numpy as np
# import time

# # Load YOLOv8 model
# model = YOLO("yolov8n.pt")

# # Initialize video capture
# cap = cv2.VideoCapture(0)
# ret, prev_frame = cap.read()

# # Optical flow params (for motion tracking)
# prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

# # For timed reporting
# moving_objects = set()
# last_report_time = time.time()

# while cap.isOpened():
#     ret, frame = cap.read()
#     if not ret:
#         break

#     # Object Detection
#     results = model(frame, verbose=False)[0]

#     # Motion Detection
#     gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
#     flow = cv2.absdiff(prev_gray, gray)
#     _, motion_mask = cv2.threshold(flow, 25, 255, cv2.THRESH_BINARY)
#     motion_mask = cv2.cvtColor(motion_mask, cv2.COLOR_GRAY2BGR)

#     # Draw boxes and check for motion in bounding boxes
#     for box in results.boxes:
#         x1, y1, x2, y2 = map(int, box.xyxy[0])
#         conf = float(box.conf[0])
#         cls = int(box.cls[0])
#         label = model.names[cls]

#         # Check if this area contains motion
#         object_motion_area = motion_mask[y1:y2, x1:x2]
#         motion_level = np.mean(object_motion_area)

#         # Threshold to consider the object as "moving"
#         moving = motion_level > 5

#         if moving:
#             moving_objects.add(label)

#         color = (0, 255, 0) if moving else (0, 0, 255)
#         cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
#         text = f"{label} ({'moving' if moving else 'still'})"
#         cv2.putText(frame, text, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

#     # Print only ONCE every 10 seconds
#     current_time = time.time()
#     if current_time - last_report_time >= 5:
#         print(f"Moving objects in the last 10 seconds: {sorted(moving_objects)}")
#         moving_objects.clear()
#         last_report_time = current_time

#     cv2.imshow("YOLO + Motion Detection", frame)
#     prev_gray = gray.copy()

#     if cv2.waitKey(1) == ord('q'):
#         break

# cap.release()
# cv2.destroyAllWindows()


Moving objects in the last 10 seconds: ['bed', 'person']
Moving objects in the last 10 seconds: ['person']
Moving objects in the last 10 seconds: ['person']
Moving objects in the last 10 seconds: ['bed', 'person']
Moving objects in the last 10 seconds: ['bed', 'chair', 'person']
Moving objects in the last 10 seconds: ['bed', 'person']
Moving objects in the last 10 seconds: ['person']
Moving objects in the last 10 seconds: ['bed', 'chair', 'person']
Moving objects in the last 10 seconds: ['bed', 'person']
Moving objects in the last 10 seconds: ['bed', 'person']
Moving objects in the last 10 seconds: ['bed', 'cell phone', 'person', 'remote']
Moving objects in the last 10 seconds: ['bed', 'person']


KeyboardInterrupt: 